In [ ]:
from pathlib import Path
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

RANDOM_STATE = 42

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
checkpoint_path = DATA_DIR / "GSE25055_pre_lasso.joblib"

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path.resolve()}")

checkpoint = joblib.load(checkpoint_path)
X = checkpoint["X"]
y = checkpoint["y"]

outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
outer_splits = list(outer_cv.split(X, y))

def create_lasso_pipeline():

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "lasso",
            LogisticRegression(
                l1_ratio=1.0,
                solver="liblinear",
                class_weight="balanced",
                max_iter=10000,
                random_state=RANDOM_STATE
            )
        )
    ])

parameter_grid = {
    "lasso__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        1.0
    ]
}

print("X:", X.shape, "| y:", y.shape, "| outer folds:", len(outer_splits))

Project root added to path: d:\diplom-project
X: (306, 22283) | y: (306,) | outer folds: 10


In [ ]:
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from src.models.custom_random_forest import (
    build_forest,
    predict_forest
)
NUMBER_OF_TREES = 30
MAX_DEPTH = 6
MIN_SAMPLES_SPLIT = 5

FOLDS_TO_RUN = 10

def calculate_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan
    }

feature_names = X.columns.to_numpy()

fold_records = []
selected_probes_by_fold = {}

custom_oof_probability = pd.Series(np.nan, index=y.index)
custom_oof_prediction = pd.Series(pd.NA, index=y.index, dtype="Int64")
sklearn_oof_probability = pd.Series(np.nan, index=y.index)
sklearn_oof_prediction = pd.Series(pd.NA, index=y.index, dtype="Int64")

experiment_start = time.time()

for fold, (train_idx, val_idx) in enumerate(outer_splits, start=1):
    if fold > FOLDS_TO_RUN:
        break

    print(f"\n=== Fold {fold}/{FOLDS_TO_RUN} ===")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # --- LASSO selection, само върху train частта ---
    search = GridSearchCV(
        estimator=create_lasso_pipeline(),
        param_grid=parameter_grid,
        scoring="average_precision",
        cv=inner_cv, refit=True, n_jobs=-1
    )
    search.fit(X_train, y_train)

    coefficients = search.best_estimator_.named_steps["lasso"].coef_.ravel()
    selected_mask = np.abs(coefficients) > 1e-10
    selected_probes = feature_names[selected_mask]
    selected_probes_by_fold[fold] = list(selected_probes)

    print(f"LASSO избра {len(selected_probes)} probes (C={search.best_params_['lasso__C']})")

    X_train_sel = X_train[selected_probes].to_numpy(dtype=float)
    X_val_sel = X_val[selected_probes].to_numpy(dtype=float)
    y_train_arr = y_train.to_numpy(dtype=int)

    # --- Custom Random Forest ---
    t0 = time.time()
    forest = build_forest(
        X_train_sel, y_train_arr,
        number_of_trees=NUMBER_OF_TREES, max_depth=MAX_DEPTH,
        min_samples_split=MIN_SAMPLES_SPLIT, random_state=RANDOM_STATE
    )
    custom_pred, custom_proba = predict_forest(forest, X_val_sel)
    custom_time = time.time() - t0

    custom_oof_probability.loc[X_val.index] = custom_proba
    custom_oof_prediction.loc[X_val.index] = custom_pred

    # --- Sklearn Random Forest (същите избрани probes, честно сравнение) ---
    t0 = time.time()
    sk_model = RandomForestClassifier(
    n_estimators=NUMBER_OF_TREES,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE + fold,
    n_jobs=-1
)
    sk_model.fit(X_train_sel, y_train_arr)
    sklearn_proba = sk_model.predict_proba(X_val_sel)[:, 1]
    sklearn_pred = sk_model.predict(X_val_sel)
    sklearn_time = time.time() - t0

    sklearn_oof_probability.loc[X_val.index] = sklearn_proba
    sklearn_oof_prediction.loc[X_val.index] = sklearn_pred

    # --- Метрики и за двата модела ---
    custom_metrics = calculate_metrics(y_val.to_numpy(), custom_pred, custom_proba)
    sklearn_metrics = calculate_metrics(y_val.to_numpy(), sklearn_pred, sklearn_proba)

    fold_records.append({"fold": fold, "model": "Custom Random Forest", "selected_probes": len(selected_probes), "rf_time_seconds": custom_time, **custom_metrics})
    fold_records.append({"fold": fold, "model": "Sklearn Random Forest", "selected_probes": len(selected_probes), "rf_time_seconds": sklearn_time, **sklearn_metrics})

    print(f"Custom  ROC-AUC={custom_metrics['roc_auc']:.3f} | Sensitivity={custom_metrics['sensitivity']:.3f} | {custom_time:.1f}s")
    print(f"Sklearn ROC-AUC={sklearn_metrics['roc_auc']:.3f} | Sensitivity={sklearn_metrics['sensitivity']:.3f} | {sklearn_time:.1f}s")

experiment_time = time.time() - experiment_start
print(f"\nОбщо време: {experiment_time:.1f}s")


=== Fold 1/10 ===


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LASSO избра 74 probes (C=0.1)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.860 | Sensitivity=0.000 | 7.5s
Sklearn ROC-AUC=0.813 | Sensitivity=0.167 | 0.0s

=== Fold 2/10 ===


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LASSO избра 750 probes (C=100.0)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.707 | Sensitivity=0.167 | 200.5s
Sklearn ROC-AUC=0.767 | Sensitivity=0.167 | 0.1s

=== Fold 3/10 ===


...

LASSO избра 757 probes (C=100.0)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.608 | Sensitivity=0.000 | 219.5s
Sklearn ROC-AUC=0.712 | Sensitivity=0.200 | 0.1s

=== Fold 9/10 ===


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LASSO избра 700 probes (C=100.0)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.936 | Sensitivity=0.000 | 182.4s
Sklearn ROC-AUC=0.864 | Sensitivity=0.000 | 0.1s

=== Fold 10/10 ===


d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\diplom-project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


LASSO избра 87 probes (C=0.1)
Building tree 1 of 30
Building tree 2 of 30
Building tree 3 of 30
Building tree 4 of 30
Building tree 5 of 30
Building tree 6 of 30
Building tree 7 of 30
Building tree 8 of 30
Building tree 9 of 30
Building tree 10 of 30
Building tree 11 of 30
Building tree 12 of 30
Building tree 13 of 30
Building tree 14 of 30
Building tree 15 of 30
Building tree 16 of 30
Building tree 17 of 30
Building tree 18 of 30
Building tree 19 of 30
Building tree 20 of 30
Building tree 21 of 30
Building tree 22 of 30
Building tree 23 of 30
Building tree 24 of 30
Building tree 25 of 30
Building tree 26 of 30
Building tree 27 of 30
Building tree 28 of 30
Building tree 29 of 30
Building tree 30 of 30
Custom  ROC-AUC=0.722 | Sensitivity=0.000 | 20.4s
Sklearn ROC-AUC=0.681 | Sensitivity=0.000 | 0.1s

Общо време: 1650.6s


In [5]:
# ============================================================
# ЗАПАЗВАНЕ НА ПИЛОТНИЯ BASELINE ЕКСПЕРИМЕНТ
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn


# ------------------------------------------------------------
# 1. Намираме основната папка на проекта
# ------------------------------------------------------------

current_directory = Path.cwd().resolve()

if (current_directory / "src").is_dir():
    project_directory = current_directory

elif (current_directory.parent / "src").is_dir():
    project_directory = current_directory.parent

else:
    raise FileNotFoundError(
        "Не намирам основната папка на проекта."
    )


# ------------------------------------------------------------
# 2. Създаваме отделна папка за пилотния експеримент
# ------------------------------------------------------------

pilot_results_directory = (
    project_directory
    / "results"
    / "lasso_custom_rf_first_run"
)

pilot_results_directory.mkdir(
    parents=True,
    exist_ok=True
)




# ------------------------------------------------------------
# 3. Проверяваме дали са изпълнени всичките 10 folds
#
# За всеки fold имаме два реда:
# 1 ред за Custom Random Forest
# 1 ред за Sklearn Random Forest
#
# Следователно очакваме общо 20 реда.
# ------------------------------------------------------------

expected_number_of_records = 20

if len(fold_records) != expected_number_of_records:
    raise RuntimeError(
        f"Очаквахме {expected_number_of_records} записа, "
        f"но са налични {len(fold_records)}."
    )


# Проверяваме дали всеки пациент има OOF прогноза.
if custom_oof_probability.isna().any():
    raise RuntimeError(
        "Липсват Custom RF вероятности за някои пациенти."
    )

if custom_oof_prediction.isna().any():
    raise RuntimeError(
        "Липсват Custom RF прогнози за някои пациенти."
    )

if sklearn_oof_probability.isna().any():
    raise RuntimeError(
        "Липсват Sklearn RF вероятности за някои пациенти."
    )

if sklearn_oof_prediction.isna().any():
    raise RuntimeError(
        "Липсват Sklearn RF прогнози за някои пациенти."
    )

print("Проверката е успешна: всички folds са завършени.")


# ------------------------------------------------------------
# 4. Създаваме таблица с резултатите по folds
# ------------------------------------------------------------

fold_results = pd.DataFrame(
    fold_records
)


# В първоначалния код не запазихме best C във fold_records.
# Затова го добавяме от изведения резултат на експеримента.
best_c_by_fold = {
    1: 0.1,
    2: 100.0,
    3: 0.1,
    4: 100.0,
    5: 100.0,
    6: 100.0,
    7: 100.0,
    8: 100.0,
    9: 100.0,
    10: 0.1
}

fold_results["lasso_best_c"] = (
    fold_results["fold"].map(
        best_c_by_fold
    )
)


# В пилотното изпълнение не сме записали кое class_weight
# е избрано от GridSearchCV за всеки fold.
#
# Не измисляме стойности, а отбелязваме честно,
# че тази информация не е запазена.
fold_results["lasso_best_class_weight"] = (
    "not_recorded"
)


# Подреждаме редовете.
fold_results = fold_results.sort_values(
    by=[
        "fold",
        "model"
    ]
).reset_index(
    drop=True
)


# Записваме таблицата.
fold_results_path = (
    pilot_results_directory
    / "fold_results.csv"
)

fold_results.to_csv(
    fold_results_path,
    index=False
)


# ------------------------------------------------------------
# 5. Изчисляваме средна стойност и стандартно отклонение
# ------------------------------------------------------------

metric_columns = [
    "selected_probes",
    "roc_auc",
    "average_precision",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "precision",
    "sensitivity",
    "specificity",
    "rf_time_seconds"
]

cv_summary = (
    fold_results
    .groupby("model")[metric_columns]
    .agg([
        "mean",
        "std"
    ])
)


# Премахваме двойното ниво на имената на колоните.
cv_summary.columns = [
    column_name + "_" + statistic
    for column_name, statistic
    in cv_summary.columns
]

cv_summary = cv_summary.reset_index()


summary_path = (
    pilot_results_directory
    / "cross_validation_summary.csv"
)

cv_summary.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# 6. Създаваме таблица с OOF прогнозите
#
# OOF = out-of-fold.
# Всеки пациент е предсказан от модел, който не е бил
# обучаван с данните на този пациент.
# ------------------------------------------------------------

oof_results = pd.DataFrame({
    "patient_id": y.index.astype(str),
    "true_label": y.to_numpy(dtype=int),

    "custom_probability": (
        custom_oof_probability
        .to_numpy(dtype=float)
    ),

    "custom_prediction": (
        custom_oof_prediction
        .astype("int64")
        .to_numpy()
    ),

    "sklearn_probability": (
        sklearn_oof_probability
        .to_numpy(dtype=float)
    ),

    "sklearn_prediction": (
        sklearn_oof_prediction
        .astype("int64")
        .to_numpy()
    )
})


oof_results_path = (
    pilot_results_directory
    / "oof_predictions.csv"
)

oof_results.to_csv(
    oof_results_path,
    index=False
)


# ------------------------------------------------------------
# 7. Изчисляваме общите OOF метрики
# ------------------------------------------------------------

def calculate_complete_metrics(
    y_true,
    y_prediction,
    y_probability
):
    """
    Изчислява метриките върху всички OOF прогнози.
    """

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            y_true,
            y_probability
        ),

        "average_precision": average_precision_score(
            y_true,
            y_probability
        ),

        "accuracy": accuracy_score(
            y_true,
            y_prediction
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_prediction
        ),

        "f1": f1_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "precision": precision_score(
            y_true,
            y_prediction,
            zero_division=0
        ),

        "sensitivity": (
            tp / (tp + fn)
            if (tp + fn) > 0
            else np.nan
        ),

        "specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0
            else np.nan
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }


true_labels = oof_results[
    "true_label"
].to_numpy()

custom_overall_metrics = calculate_complete_metrics(
    true_labels,
    oof_results["custom_prediction"].to_numpy(),
    oof_results["custom_probability"].to_numpy()
)

sklearn_overall_metrics = calculate_complete_metrics(
    true_labels,
    oof_results["sklearn_prediction"].to_numpy(),
    oof_results["sklearn_probability"].to_numpy()
)


overall_results = pd.DataFrame([
    {
        "model": "Custom Random Forest",
        **custom_overall_metrics
    },
    {
        "model": "Sklearn Random Forest",
        **sklearn_overall_metrics
    }
])


overall_results_path = (
    pilot_results_directory
    / "overall_oof_results.csv"
)

overall_results.to_csv(
    overall_results_path,
    index=False
)


# ------------------------------------------------------------
# 8. Запазваме избраните probes за всеки fold
# ------------------------------------------------------------

selected_probe_records = []

for fold, selected_probes in selected_probes_by_fold.items():

    for probe_id in selected_probes:

        selected_probe_records.append({
            "fold": fold,
            "probe_id": probe_id
        })


selected_probes_table = pd.DataFrame(
    selected_probe_records
)


selected_probes_path = (
    pilot_results_directory
    / "selected_probes_by_fold.csv"
)

selected_probes_table.to_csv(
    selected_probes_path,
    index=False
)


# ------------------------------------------------------------
# 9. Запазваме пълен checkpoint
#
# CSV файловете са удобни за разглеждане.
# Joblib checkpoint-ът позволява резултатите по-късно
# да бъдат заредени обратно в Python.
# ------------------------------------------------------------

pilot_checkpoint = {
    "experiment_name": "lasso_custom_rf_first_run",

    "description": (
        "Fold-specific LASSO followed by Custom and "
        "Sklearn Random Forest. Fixed classification "
        "threshold 0.5 and no RF class-imbalance correction."
    ),

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "dataset_id": "GSE25055",

    "external_dataset_used": False,

    "random_state": RANDOM_STATE,

    "outer_folds": 10,
    "inner_folds": 5,

    "number_of_trees": NUMBER_OF_TREES,
    "max_depth": MAX_DEPTH,
    "min_samples_split": MIN_SAMPLES_SPLIT,

    "classification_threshold": 0.5,

    "lasso_parameter_grid": parameter_grid,

    "best_c_by_fold": best_c_by_fold,

    "lasso_best_class_weight_note": (
        "Not recorded separately during the pilot run."
    ),

    "fold_results": fold_results,
    "cv_summary": cv_summary,
    "overall_oof_results": overall_results,
    "oof_predictions": oof_results,

    "selected_probes_by_fold": (
        selected_probes_by_fold
    ),

    "outer_splits": outer_splits,

    "experiment_time_seconds": experiment_time,

    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__
}


checkpoint_path = (
    pilot_results_directory
    / "lasso_custom_rf_first_run.joblib"
)

joblib.dump(
    pilot_checkpoint,
    checkpoint_path
)


# ------------------------------------------------------------
# 10. Показваме резултата
# ------------------------------------------------------------

print("\nПилотният експеримент е запазен успешно.")

print("\nРезултати по folds:")
display(fold_results)

print("\nСредни стойности и стандартни отклонения:")
display(cv_summary)

print("\nОбщи OOF резултати:")
display(overall_results)

print("\nСъздадени файлове:")

for saved_file in sorted(
    pilot_results_directory.iterdir()
):
    print("-", saved_file.name)

Проверката е успешна: всички folds са завършени.

Пилотният експеримент е запазен успешно.

Резултати по folds:


,fold,model,selected_probes,rf_time_seconds,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,lasso_best_c,lasso_best_class_weight
0,1,Custom Random Forest,74,7.523554,0.860000,0.552673,0.774194,0.480000,0.000000,0.0,0.000000,0.960000,0.1,not_recorded
1,1,Sklearn Random Forest,74,0.043432,0.813333,0.504762,0.806452,0.563333,0.250000,0.5,0.166667,0.960000,0.1,not_recorded
2,2,Custom Random Forest,750,200.549500,0.706667,0.443754,0.806452,0.563333,0.250000,0.5,0.166667,0.960000,100.0,not_recorded
3,2,Sklearn Random Forest,750,0.079513,0.766667,0.592816,0.838710,0.583333,0.285714,1.0,0.166667,1.000000,100.0,not_recorded
4,3,Custom Random Forest,77,9.134013,0.733333,0.451703,0.741935,0.460000,0.000000,0.0,0.000000,0.920000,0.1,not_recorded
5,3,Sklearn Random Forest,77,0.047992,0.760000,0.631349,0.870968,0.666667,0.500000,1.0,0.333333,1.000000,0.1,not_recorded
6,4,Custom Random Forest,726,161.759546,0.660000,0.351706,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded
7,4,Sklearn Random Forest,726,0.079089,0.753333,0.501578,0.838710,0.583333,0.285714,1.0,0.166667,1.000000,100.0,not_recorded
8,5,Custom Random Forest,661,142.445101,0.753333,0.391880,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded
9,5,Sklearn Random Forest,661,0.094738,0.846667,0.541967,0.806452,0.500000,0.000000,0.0,0.000000,1.000000,100.0,not_recorded



Средни стойности и стандартни отклонения:


,model,selected_probes_mean,selected_probes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,...,f1_mean,f1_std,precision_mean,precision_std,sensitivity_mean,sensitivity_std,specificity_mean,specificity_std,rf_time_seconds_mean,rf_time_seconds_std
0,Custom Random Forest,525.4,310.523286,0.743689,0.103595,0.451064,0.129900,0.804086,0.029280,0.506667,...,0.053571,0.113252,0.15,0.337474,0.033333,0.070273,0.980000,0.028284,135.431495,87.951518
1,Sklearn Random Forest,525.4,310.523286,0.778989,0.060291,0.518811,0.082572,0.823441,0.032197,0.545583,...,0.165476,0.186504,0.45,0.497214,0.103333,0.119102,0.987833,0.019595,0.085458,0.027916



Общи OOF резултати:


,model,roc_auc,average_precision,accuracy,balanced_accuracy,f1,precision,sensitivity,specificity,true_negative,false_positive,false_negative,true_positive
0,Custom Random Forest,0.748820,0.402658,0.803922,0.507504,0.062500,0.285714,0.035088,0.979920,244,5,55,2
1,Sklearn Random Forest,0.775594,0.452917,0.823529,0.546607,0.181818,0.666667,0.105263,0.987952,246,3,51,6



Създадени файлове:
- cross_validation_summary.csv
- fold_results.csv
- lasso_custom_rf_first_run.joblib
- oof_predictions.csv
- overall_oof_results.csv
- selected_probes_by_fold.csv
